# AvgPool_VGG-16 hypertension staging from PPG — quickstart

Replication of arXiv:2304.06952 plus a stronger 1-D hybrid, on the PPG-BP dataset.

**Before you start:** enable a GPU.
- Colab: `Runtime → Change runtime type → T4 GPU`
- Kaggle: `Settings → Accelerator → GPU`, and **Internet ON**

Full documentation is in `GUIDE.md`.

## 1. Locate the code

Upload and unpack `hyperppg_code.zip`, then `cd` into it. Adjust the path if you
put the folder somewhere else (e.g. Google Drive).

In [ ]:
import os, glob, zipfile, sys

# Unpack an uploaded zip if present.
for z in glob.glob('/content/*.zip') + glob.glob('/kaggle/working/*.zip'):
    if 'pretrain' not in os.path.basename(z):
        zipfile.ZipFile(z).extractall(os.path.dirname(z))
        print('extracted', z)

# Find the package root (the directory containing 'hyperppg').
root = None
for base in ('/content', '/kaggle/working', '.'):
    for cand in glob.glob(os.path.join(base, '**', 'hyperppg'), recursive=True):
        if os.path.isdir(cand) and os.path.isfile(os.path.join(cand, 'config.py')):
            root = os.path.dirname(cand)
            break
    if root:
        break

assert root, 'could not find the hyperppg package -- upload the code first'
os.chdir(root)
sys.path.insert(0, root)
print('working directory:', os.getcwd())
print(sorted(os.listdir()))

## 2. Dependencies

Torch is already installed on Colab and Kaggle with the right CUDA build.
**Do not reinstall it** — that breaks the GPU runtime.

In [ ]:
!pip install -q openpyxl lightgbm

import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## 3. Download PPG-BP (1.5 MB)

In [ ]:
!python -m hyperppg.data.download --dest data/ppgbp

## 4. Validate the pipeline

Expect `14/14 checks passed`. If anything fails, fix it before training —
the message names the exact stage.

In [ ]:
!python -m hyperppg.selfcheck

## 5. Look at the data

Raw segment, the paper's preprocessing, and the rendered image the CNNs see.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from hyperppg.data import ppgbp
from hyperppg.data.preprocess import paper_pipeline, clean_pipeline
from hyperppg.data.render import waveform_to_image
from hyperppg.config import CLASS_NAMES

index = ppgbp.build_index()
print(ppgbp.summarise(index))

X = ppgbp.load_signals(index)
Xp = paper_pipeline(X)
Xc = clean_pipeline(X)

# One example per class.
fig, axes = plt.subplots(4, 3, figsize=(14, 9))
for r, cls in enumerate(range(4)):
    i = int(np.where(index['y'].to_numpy() == cls)[0][0])
    axes[r, 0].plot(X[i], lw=0.7)
    axes[r, 0].set_ylabel(CLASS_NAMES[cls][:14], fontsize=9)
    axes[r, 1].plot(Xc[i], lw=0.8, color='tab:green')
    img = waveform_to_image(Xp[i], normalize=False)
    axes[r, 2].imshow(img[0], cmap='gray')
    axes[r, 2].axis('off')
    if r == 0:
        axes[r, 0].set_title('raw (1000 Hz)')
        axes[r, 1].set_title('cleaned (125 Hz)')
        axes[r, 2].set_title('rendered image (CNN input)')
plt.tight_layout()
plt.show()

## 6. See the leakage for yourself

PPG-BP has 3 segments per subject. Splitting on segments rather than subjects
puts the same person on both sides of the split.

In [ ]:
from hyperppg.data.splits import make_folds, describe_folds

for scheme in ('subject', 'segment'):
    print(f'--- {scheme} split ---')
    print(describe_folds(index, make_folds(index, scheme=scheme, n_splits=5)))
    print()

## 7. Classical baseline (~1 min, CPU)

63 morphology features + gradient boosting. This is the bar every deep model
must clear. Running both splits shows the leakage gap numerically.

In [ ]:
!python -m hyperppg.features_baseline --split both

## 8. Replicate the paper

AvgPool_VGG-16 under both protocols. Use `--model all` to include AlexNet,
ResNet-50 and plain VGG-16 as well (slower).

In [ ]:
!python -m hyperppg.train_paper --model avgpool_vgg16 --split both --epochs 30 --batch-size 32

## 9. The improved model

1-D CNN + transformer on the signal itself: 2.2 M parameters against VGG-16's
134 M.

In [ ]:
!python -m hyperppg.train_hybrid --folds 5 --epochs 80

## 10. Optional: self-supervised pretraining on PPG-DaLiA + FatigueSet

Build `ppg_pretrain.zip` on your Windows machine first:

```powershell
E:\ppg\hypertension_avgpool_vgg16\scripts\make_pretrain_bundle.ps1
```

That turns ~3.5 GB into ~50 MB. Upload it, then run the two cells below.

In [ ]:
import glob, zipfile, os

hits = glob.glob('/content/**/ppg_pretrain.zip', recursive=True) + \
       glob.glob('/kaggle/**/ppg_pretrain.zip', recursive=True)
if hits:
    zipfile.ZipFile(hits[0]).extractall('pretrain')
    print('extracted to pretrain/:', sorted(os.listdir('pretrain')))
else:
    print('ppg_pretrain.zip not found -- upload it to run this section')

In [ ]:
!python -m hyperppg.pretrain_ssl \
    --dalia pretrain/dalia --fatigueset pretrain/fatigueset \
    --max-windows 120000 --epochs 30 --cache corpus.npy --out runs/ssl

In [ ]:
# Fine-tune. Watch for "loaded 104 encoder tensors" -- that confirms transfer.
!python -m hyperppg.train_hybrid --ssl-checkpoint runs/ssl/ssl_encoder.pt --folds 5 --epochs 80

## 11. Ensemble the saved out-of-fold logits

Every run writes `*_oof_logits.npy`, so you can combine runs without retraining.
Re-run sections 9/10 with different `--seed` values first for this to help.

In [ ]:
import glob
import numpy as np

from hyperppg.metrics import format_report

paths = sorted(glob.glob('**/*_oof_logits.npy', recursive=True))
print(f'found {len(paths)} logit files')
for p in paths:
    print(' ', p)

if paths:
    y = index['y'].to_numpy()
    # Softmax each run before averaging so differently-scaled logits combine fairly.
    probs = []
    for p in paths:
        z = np.load(p)
        z = z - z.max(axis=1, keepdims=True)
        e = np.exp(z)
        probs.append(e / e.sum(axis=1, keepdims=True))
    pred = np.mean(probs, axis=0).argmax(axis=1)
    print()
    print(format_report(y, pred, title=f'ensemble of {len(paths)} runs'))